# Shell-Restricted Parameter Marker Correlations

Extracted from `embeddings_parameters.ipynb`. This notebook contains the shell-restricted parameter-marker correlation analysis and its downstream summaries.

## Shell-restricted parameter-marker correlations

For each sparse-PCA parameter direction (D1-D3), restrict markers to gene neighbourhood shells
from two methods (STRING, OmniPath Interactions)
around the corresponding seed gene(s):

| Group | Anchor parameter | Seed gene(s) |
|-------|-----------------|--------------|
| D1 (SC1, 38 % var) | ERBB2_kw + ERBB2_EGFR | ERBB2 |
| D2 (SC2, 30 % var) | RPS6KA1_kw + iMEK + MEK_ERBB2 | RPS6KA1, RPS6KA2, RPS6KA3, RPS6KA6, MAP2K1, MAP2K2 |
| D3 (SC3, 22 % var) | ERK_MEK_kw + MEK_EGFR + iMEK | MAPK1, MAPK3, MAP2K1, MAP2K2 |

BH FDR correction is applied per parameter group x shell scope.
Three visual states in heatmaps:
- **Coloured** = significant (q < 0.05)
- **x marker** = tested but not significant
- **Grey** = gene not in that method's shell

In [ ]:
# --- Fetch gene neighbourhood shells for parameter groups D1-D3 ----------------
import sys, importlib
from pathlib import Path

_project_root = str(Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import gene_shells as _gs
importlib.reload(_gs)
from gene_shells import get_shells, load_msigdb_gmt, ShellResult

# Parameter group -> seed genes
# D1 (SC1): ERBB2 signalling axis
# D2 (SC2): RSK / MEK-ERBB2 / iMEK axis
# D3 (SC3): ERK-MEK / MEK-EGFR / iMEK axis
PARAM_TO_SEEDS = {
    "D1": ["ERBB2"],
    "D2": ["RPS6KA1", "RPS6KA2", "RPS6KA3", "RPS6KA6", "MAP2K1", "MAP2K2"],
    "D3": ["MAPK1", "MAPK3", "MAP2K1", "MAP2K2"],
}

METHOD_CONFIGS = {
    "STRING (>=900)":          dict(method="string", score=900),
    "OmniPath Interactions":  dict(method="omnipath_interactions"),
}

# Fetch shells: shell_results[param_group][method_label] = ShellResult
shell_results: dict[str, dict[str, ShellResult]] = {}
for pg, seeds in PARAM_TO_SEEDS.items():
    shell_results[pg] = {}
    for label, kw in METHOD_CONFIGS.items():
        res = get_shells(seeds, **kw)
        shell_results[pg][label] = res
        s = res.summary()
        print(f"  {pg} | {label:25s} | "
              f"{s['n_seeds']} seeds + {s['n_shell1']:>5,} S1 + "
              f"{s['n_shell2']:>5,} S2 = {s['n_all']:>6,} total")

print(f"\nDone: {len(shell_results)} param groups x {len(METHOD_CONFIGS)} methods")

In [ ]:
# --- Compute shell-restricted Spearman correlations: parameter group vs markers ---
# Per-job approach: for each training job, compute the parameter gradient from
# that job's parameter deviations, project the (shared) PCA embeddings, then
# correlate with markers.  Report median rho/p across jobs (+ std).
import numpy as np
import pandas as pd
import importlib
from sklearn.linear_model import LinearRegression
from embeddings import (
    load_all_marker_data,
    load_embedding_data,
    load_marcotte_subtypes,
    prepare_pca_embeddings,
    compute_averaged_parameter_gradient,
    sensitive_dirs,
)
from common import evaluations_dir
import figures_paper.shell_correlations as _sc
importlib.reload(_sc)
from figures_paper.shell_correlations import shell_restricted_correlations

MODEL = "EGFR_MAPK__logobs_tegfr_aggavg"
CONTEXT = "cytof_init"
FIGURE = "figure3"
DATA = "dream_cytof"
PARAM_GROUPS = ["D1", "D2", "D3"]
UNION_LABEL = "Union (all)"

# --- Load PCA embeddings (already aggregated across jobs) --------------------
embedding_df = load_embedding_data(FIGURE, DATA)
subtypes_df = load_marcotte_subtypes(embedding_df.cell_line.unique())
pca_emb = prepare_pca_embeddings(embedding_df, subtypes_df)
emb_df = pca_emb[(pca_emb["model"] == MODEL) & (pca_emb["context"] == CONTEXT)].copy()

embedding_cols = [c for c in emb_df.columns if c.startswith("L")]
embeddings_array = emb_df[embedding_cols].values

# --- Load per-job parameter deviations --------------------------------------
_param_path = evaluations_dir / MODEL / DATA / f"param_devs_{FIGURE}.csv"
_param_raw = pd.read_csv(_param_path, index_col=0)
_param_raw = _param_raw[
    (_param_raw["context"] == CONTEXT) &
    (_param_raw["samples"].str.startswith("all"))
]
job_ids = sorted(_param_raw["job"].unique())
n_jobs = len(job_ids)
print(f"Jobs: {n_jobs}  ({job_ids[:3]}{'...' if n_jobs > 3 else ''})")

# --- Compute per-job parameter projections -----------------------------------
# For each job: compute per-parameter gradient via LinearRegression, average,
# normalise, project embeddings -> one Series(cell_line -> projection) per job.
param_per_job: dict[str, dict] = {}
param_values_avg: dict[str, pd.Series] = {}

for pg in PARAM_GROUPS:
    param_per_job[pg] = {}

    for job_id in job_ids:
        job_df = _param_raw[_param_raw["job"] == job_id]
        gradients = []
        for param in sensitive_dirs[pg]:
            if param not in job_df.columns:
                continue
            pv = job_df.set_index("cell_line")[param]
            common = emb_df.index.intersection(pv.index)
            pv_sub = pv.loc[common]
            valid = pv_sub.notna()
            if valid.sum() < 5:
                continue
            X = emb_df.loc[common].loc[valid, embedding_cols].values
            y = pv_sub.loc[valid].values
            reg = LinearRegression().fit(X, y)
            gradients.append(reg.coef_)

        if not gradients:
            continue
        avg_grad = np.mean(gradients, axis=0)
        avg_grad = avg_grad / np.linalg.norm(avg_grad)
        proj = pd.Series(embeddings_array @ avg_grad, index=emb_df.index)
        param_per_job[pg][job_id] = proj

    _, proj_avg = compute_averaged_parameter_gradient(
        model=MODEL, context=CONTEXT, parameter_group=sensitive_dirs[pg],
        embedding_df=emb_df, figure=FIGURE, data=DATA,
    )
    param_values_avg[pg] = proj_avg
    print(f"  {pg}: {len(param_per_job[pg])}/{n_jobs} jobs with valid projections")

param_df = pd.DataFrame(param_values_avg)
print(f"\nParameter projections: {param_df.shape[0]} cell lines x {param_df.shape[1]} groups")

# --- Load marker data --------------------------------------------------------
prot_all = load_all_marker_data("proteomics")
tx_all   = load_all_marker_data("transcriptomics")

N_HVG = 8000

# --- Run per-job shell-restricted correlations (per-method + union) ----------
all_sig_rows = []
for pg in PARAM_GROUPS:
    metric_jobs = param_per_job[pg]
    if not metric_jobs:
        continue

    all_seeds = set()
    for ml in METHOD_CONFIGS:
        all_seeds |= set(shell_results[pg][ml].seeds)

    for dtype_label, md_full, n_hvg in [
        ("proteomics", prot_all, None),
        ("transcriptomics", tx_all, N_HVG),
    ]:
        for method_label in METHOD_CONFIGS:
            sr = shell_results[pg][method_label]
            df = shell_restricted_correlations(
                metric_jobs, md_full,
                shell_genes=set(sr.seed_and_shell1),
                seeds_set=set(sr.seeds),
                data_type_label=dtype_label,
                n_hvg=n_hvg,
            )
            if df.empty:
                continue
            sig = df[df["significant"]].copy()
            if not sig.empty:
                sig["param_group"] = pg
                sig["shell_method"] = method_label
                all_sig_rows.append(sig)

        union_genes = set()
        for ml in METHOD_CONFIGS:
            union_genes |= set(shell_results[pg][ml].seed_and_shell1)

        df_u = shell_restricted_correlations(
            metric_jobs, md_full,
            shell_genes=union_genes,
            seeds_set=all_seeds,
            data_type_label=dtype_label,
            n_hvg=n_hvg,
        )
        if not df_u.empty:
            sig_u = df_u[df_u["significant"]].copy()
            if not sig_u.empty:
                sig_u["param_group"] = pg
                sig_u["shell_method"] = UNION_LABEL
                all_sig_rows.append(sig_u)

sig_df = pd.concat(all_sig_rows, ignore_index=True) if all_sig_rows else pd.DataFrame()
print(f"\nTotal significant hits: {len(sig_df)}")
if len(sig_df) > 0:
    print(f"\nBreakdown by shell method x data type:")
    print(sig_df.groupby(["shell_method", "data_type"]).size()
          .unstack(fill_value=0).to_string())
    n_seed_hits = (sig_df["shell"] == "seed").sum()
    n_s1_hits = (sig_df["shell"] == "S1").sum()
    print(f"\nOf these: {n_seed_hits} seeds, {n_s1_hits} S1")

In [ ]:
# --- Display significant hits table -------------------------------------------
if len(sig_df) > 0:
    display_cols = ["param_group", "shell_method", "data_type",
                    "marker", "shell", "rho", "rho_std", "pval", "pval_std",
                    "qval", "n_tested", "n_jobs"]
    fmt = {
        "rho": "{:+.3f}", "rho_std": "{:.3f}",
        "pval": "{:.2e}", "pval_std": "{:.2e}",
        "qval": "{:.2e}",
    }
    styled = (
        sig_df[display_cols]
        .sort_values(["param_group", "shell_method", "data_type", "qval"])
        .reset_index(drop=True)
        .style.format(fmt)
        .background_gradient(subset=["rho"], cmap="RdBu_r", vmin=-0.6, vmax=0.6)
    )
    display(styled)
else:
    print("No statistically significant hits found.")

In [ ]:
# --- Heatmap: seeds + S1, STRING + OP Interactions + Union column --------
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

ALPHA = 0.05

if len(sig_df) > 0:
    method_labels_list = list(METHOD_CONFIGS.keys()) + [UNION_LABEL]
    short_method = {m: m.split("(")[0].strip() if "(" in m else m.replace("OmniPath ", "OP ")
                    for m in method_labels_list}
    short_method[UNION_LABEL] = UNION_LABEL
    n_methods = len(method_labels_list)

    _n_jobs_col = sig_df["n_jobs"].max()

    rho_lookup = {}
    for _, row in sig_df.iterrows():
        rho_lookup[(row["marker"], row["param_group"], row["shell_method"],
                    row["data_type"])] = row["rho"]

    scope_genes = {}
    for pg in PARAM_GROUPS:
        union_scope = set()
        for ml in METHOD_CONFIGS:
            sr = shell_results[pg][ml]
            sg = set(sr.seed_and_shell1)
            scope_genes[(pg, ml)] = sg
            union_scope |= sg
        scope_genes[(pg, UNION_LABEL)] = union_scope

    for dtype_label in ["proteomics", "transcriptomics"]:
        sub = sig_df[sig_df["data_type"] == dtype_label]
        if sub.empty:
            continue

        sig_markers = sorted(sub["marker"].unique())
        if not sig_markers:
            continue

        seeds_all = set()
        for pg in PARAM_GROUPS:
            for ml in METHOD_CONFIGS:
                seeds_all |= set(shell_results[pg][ml].seeds)
        md_raw = prot_all if dtype_label == "proteomics" else tx_all
        nhvg = N_HVG if dtype_label == "transcriptomics" else None
        md_cols = set(_sc._hvg_filter(md_raw, seeds_all, nhvg).columns)

        col_tuples, rho_data, in_shell = [], {}, {}

        for pg in PARAM_GROUPS:
            for ml in method_labels_list:
                ck = (pg, ml)
                col_tuples.append(ck)
                rho_data[ck], in_shell[ck] = {}, {}
                for m in sig_markers:
                    if m in scope_genes[(pg, ml)] and m in md_cols:
                        in_shell[ck][m] = True
                        rho_data[ck][m] = rho_lookup.get((m, pg, ml, dtype_label), np.nan)
                    else:
                        in_shell[ck][m] = False
                        rho_data[ck][m] = np.nan

        rho_mat = pd.DataFrame(rho_data, index=sig_markers)
        in_shell_mat = pd.DataFrame(in_shell, index=sig_markers)

        display_cols = [(pg, short_method[ml]) for pg, ml in col_tuples]
        rho_display = rho_mat.copy()
        rho_display.columns = pd.MultiIndex.from_tuples(display_cols, names=["param group", "method"])
        in_shell_display = in_shell_mat.copy()
        in_shell_display.columns = rho_display.columns

        max_abs = rho_display.abs().max(axis=1).sort_values(ascending=False)
        rho_display = rho_display.loc[max_abs.index]
        in_shell_display = in_shell_display.loc[max_abs.index]

        sig_lookup = set()
        for _, row in sub.iterrows():
            sig_lookup.add((row["marker"], row["param_group"], short_method[row["shell_method"]]))

        n_markers = len(rho_display)
        n_cols = len(col_tuples)

        fig, ax = plt.subplots(figsize=(max(10, n_cols * 0.9), max(6, n_markers * 0.35)))
        sns.heatmap(
            rho_display, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
            linewidths=0.3, linecolor="white", mask=~in_shell_display,
            cbar_kws={"label": f"median Spearman rho across {_n_jobs_col} jobs", "shrink": 0.5}, ax=ax
        )

        for i in range(n_markers):
            mk = rho_display.index[i]
            for j in range(n_cols):
                pg_label, ml = rho_display.columns[j]
                if not in_shell_display.iloc[i, j]:
                    ax.add_patch(Rectangle((j, i), 1, 1, fill=True, facecolor="#e0e0e0", edgecolor="white", linewidth=0.3))
                elif (mk, pg_label, ml) not in sig_lookup:
                    ax.text(j + 0.5, i + 0.5, "x", ha="center", va="center", fontsize=20, color="black", alpha=0.5)

        for k in range(1, len(PARAM_GROUPS)):
            ax.axvline(x=k * n_methods, color="black", linewidth=1.2)

        ax.set_title(
            f"Parameter-{dtype_label} correlations - seeds + S1\n"
            f"(median rho across {_n_jobs_col} jobs, STRING + OP Interactions + Union, HVG={N_HVG} for tx, FDR q < 0.05)",
            fontsize=12,
        )
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelsize=8)
        ax.tick_params(axis="x", labelsize=7, rotation=45)

        ax.legend(handles=[
            Rectangle((0, 0), 1, 1, facecolor="#d73027", edgecolor="none", label="rho (significant, q < 0.05)"),
            Line2D([0], [0], marker="x", color="grey", markeredgecolor="black", markersize=6, linestyle="None", alpha=0.5, label="tested, not significant"),
            Rectangle((0, 0), 1, 1, facecolor="#e0e0e0", edgecolor="white", label="not in gene list"),
        ], loc="upper left", bbox_to_anchor=(1.08, 1.0), fontsize=7, frameon=True, title="Cell meaning", title_fontsize=8)

        fig.tight_layout()
        plt.show()
        print(f"  {n_markers} markers shown")
else:
    print("No significant hits to plot.")

In [ ]:
# --- Summary: n significant per param group x method x data type --------
if len(sig_df) > 0:
    summary = (
        sig_df.groupby(["param_group", "shell_method", "data_type"])
        .size().reset_index(name="n_significant")
    )
    print("=" * 70)
    print("  Number of FDR-significant markers (q < 0.05) - shell 1 only")
    print(f"  (median rho across {sig_df['n_jobs'].max()} training jobs)")
    print("=" * 70)
    tbl = summary.pivot_table(
        index=["param_group", "shell_method"],
        columns="data_type",
        values="n_significant",
        fill_value=0,
    ).astype(int)
    display(tbl)

    print(f"\n{'=' * 70}")
    print(f"  Top-5 strongest significant correlations per parameter group")
    print(f"{'=' * 70}")
    for pg in PARAM_GROUPS:
        pg_sub = sig_df[sig_df["param_group"] == pg].nlargest(5, "abs_rho")
        if pg_sub.empty:
            print(f"  {pg}: no significant hits")
            continue
        print(f"  {pg} ({', '.join(PARAM_TO_SEEDS[pg])}):")
        for _, row in pg_sub.iterrows():
            seed_tag = " *" if row["shell"] == "seed" else ""
            rho_str = f"rho={row['rho']:+.3f}+-{row['rho_std']:.3f}"
            print(
                f"    {row['marker']:15s}  {rho_str}  q={row['qval']:.2e}  "
                f"[{row['data_type']}, {row['shell_method']}, {row['shell']}]"
                f"{seed_tag}"
            )
else:
    print("No significant hits found.")

In [ ]:
# Where does MAP3K8 rank in HVG variance?
gene = "MAP3K8"
var_all = tx_all.var().sort_values(ascending=False)
rank = (var_all.index.get_loc(gene)) + 1
print(f"MAP3K8 variance rank: {rank} / {len(var_all)}  (variance = {var_all[gene]:.4f})")
print(f"HVG cutoff at N_HVG={N_HVG}: variance >= {var_all.iloc[N_HVG-1]:.4f}")
print(f"MAP3K8 variance: {var_all[gene]:.4f}  ->  {'PASSES' if rank <= N_HVG else 'EXCLUDED'}")

gene2 = "MAPK1"
if gene2 in var_all.index:
    rank2 = (var_all.index.get_loc(gene2)) + 1
    print(f"\nMAPK1 variance rank: {rank2} / {len(var_all)}")

sr_op = shell_results["D2"]["OmniPath Interactions"]
d2_op_s1 = set(sr_op.shell1)
md_filt = _sc._hvg_filter(tx_all, set(sr_op.seeds), N_HVG)
surviving = d2_op_s1 & set(md_filt.columns)
in_tx_but_excluded = d2_op_s1 & set(tx_all.columns) - set(md_filt.columns)
print(f"\nD2 OP Interactions S1: {len(d2_op_s1)} genes")
print(f"  In tx_all: {len(d2_op_s1 & set(tx_all.columns))}")
print(f"  Survive HVG: {len(surviving)}")
print(f"  In tx_all but excluded by HVG: {len(in_tx_but_excluded)}")
if in_tx_but_excluded:
    print(f"  Excluded genes: {sorted(in_tx_but_excluded)[:20]}")